# JupyterLite（xeus-r）で学ぶ 統計テスト演習（R）

このノートブックは、ブラウザだけで動作する JupyterLite（R カーネル：xeus-r）を使って、
代表的な統計テストを体験するための演習用教材です。

扱う内容：

1. データ生成と確認
2. t検定（1標本／対応のない2標本／対応あり2標本）
3. カイ二乗（χ²）検定
4. 割合の検定（prop.test, binom.test）
5. 分散分析（ANOVA）
6. 相関分析（cor.test）
7. 回帰分析（lm）

※ すべて R の標準パッケージ（主に `stats`）のみで実行できます。

## 1. データ生成と確認

ここでは、演習用の **人工データ（乱数）** を作成します。

- テストの点数（2つのグループ）
- カテゴリデータ（性別 × 合否）


In [1]:
# 乱数のシード（毎回同じ結果が出るように）
set.seed(123)

# グループ（1: 従来法, 2: 新しい学習法）
group <- factor(rep(c("従来法", "新しい学習法"), each = 30))

# 各グループのテスト得点（正規分布から乱数生成）
score <- c(
  rnorm(30, mean = 70, sd = 10),  # 従来法
  rnorm(30, mean = 75, sd = 10)   # 新しい学習法
)

# データフレームとしてまとめる
dat <- data.frame(group = group, score = score)

# 先頭を確認
head(dat)

,group,score
,<fct>,<dbl>
1,従来法,64.39524
2,従来法,67.69823
3,従来法,85.58708
4,従来法,70.70508
5,従来法,71.29288
6,従来法,87.15065


## 2. t検定（平均の差の検定）

### 2-1. 1標本 t検定

「得点の平均が 70 と等しいかどうか」を検定します。

In [2]:
# 1標本 t検定（全体平均が 70 と等しいか）
t.test(dat$score, mu = 70)


	One Sample t-test

data:  dat$score
t = 2.5087, df = 59, p-value = 0.01488
alternative hypothesis: true mean is not equal to 70
95 percent confidence interval:
 70.63876 75.67359
sample estimates:
mean of x 
 73.15617 


### 2-2. 対応のない 2標本 t検定

従来法グループと新しい学習法グループの平均点に差があるかどうかを検定します。

In [3]:
# 対応のない2標本 t検定
t.test(score ~ group, data = dat)


	Welch Two Sample t-test

data:  score by group
t = -3.0841, df = 56.559, p-value = 0.003156
alternative hypothesis: true difference in means between group 従来法 and group 新しい学習法 is not equal to 0
95 percent confidence interval:
 -11.965426  -2.543416
sample estimates:
      mean in group 従来法 mean in group 新しい学習法 
                  69.52896                   76.78338 


### 2-3. 対応のある t検定

同一人物の「前後」を比較したい場合は、対応のある t検定を使います。

In [4]:
# 対応のある t検定の例（人工データ）

set.seed(456)
before <- rnorm(30, mean = 60, sd = 5)
after  <- before + rnorm(30, mean = 3, sd = 5)  # 介入後に少し点数が上がるイメージ

t.test(before, after, paired = TRUE)


	Paired t-test

data:  before and after
t = -4.9165, df = 29, p-value = 3.199e-05
alternative hypothesis: true mean difference is not equal to 0
95 percent confidence interval:
 -5.353462 -2.207973
sample estimates:
mean difference 
      -3.780718 


## 3. カイ二乗（χ²）検定

カテゴリ変数同士の関係を検定するのに用います。

ここでは「性別（M/F）」と「合否（Pass/Fail）」の独立性を検定します。

In [5]:
set.seed(789)

gender <- factor(sample(c("M", "F"), size = 60, replace = TRUE))
pass   <- factor(sample(c("Pass", "Fail"), size = 60, replace = TRUE))

tbl <- table(gender, pass)
tbl

      pass
gender Fail Pass
     F   20   16
     M   14   10

In [6]:
# カイ二乗検定
chisq.test(tbl)


	Pearson's Chi-squared test with Yates' continuity correction

data:  tbl
X-squared = 7.377e-31, df = 1, p-value = 1


サンプルサイズが小さいときは、Fisher の正確検定を使うこともあります。

In [7]:
# Fisher の正確検定（サンプルが小さい場合など）
fisher.test(tbl)


	Fisher's Exact Test for Count Data

data:  tbl
p-value = 1
alternative hypothesis: true odds ratio is not equal to 1
95 percent confidence interval:
 0.2746653 2.8595624
sample estimates:
odds ratio 
 0.8945442 


## 4. 割合の検定（prop.test, binom.test）

### 4-1. 2群の割合の差の検定（prop.test）

In [8]:
# 例：A 施策 40/100 人成功, B 施策 30/100 人成功
prop.test(c(40, 30), c(100, 100))


	2-sample test for equality of proportions with continuity correction

data:  c(40, 30) out of c(100, 100)
X-squared = 1.7802, df = 1, p-value = 0.1821
alternative hypothesis: two.sided
95 percent confidence interval:
 -0.04147838  0.24147838
sample estimates:
prop 1 prop 2 
   0.4    0.3 


### 4-2. 二項検定（binom.test）

「成功率が理論値 p と等しいか」を検定します。

In [9]:
# 例：80 回中 45 回成功。理論上の成功確率は 0.5 と仮定
binom.test(45, 80, p = 0.5)


	Exact binomial test

data:  45 and 80
number of successes = 45, number of trials = 80, p-value = 0.3143
alternative hypothesis: true probability of success is not equal to 0.5
95 percent confidence interval:
 0.4469976 0.6732360
sample estimates:
probability of success 
                0.5625 


## 5. 分散分析（ANOVA）

3群以上の平均の差を一度に比較するときに用います。

In [10]:
set.seed(101)

group3 <- factor(rep(c("A", "B", "C"), each = 20))
score3 <- c(
  rnorm(20, mean = 70, sd = 8),
  rnorm(20, mean = 75, sd = 8),
  rnorm(20, mean = 80, sd = 8)
)

dat3 <- data.frame(group3 = group3, score3 = score3)

# 1要因分散分析
fit_aov <- aov(score3 ~ group3, data = dat3)
summary(fit_aov)

            Df Sum Sq Mean Sq F value  Pr(>F)   
group3       2    817   408.4   7.233 0.00159 **
Residuals   57   3218    56.5                   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

## 6. 相関分析（cor.test）

2つの量的変数の線形関係の強さをみる統計量（相関係数）を推定し、その有意性を検定します。

In [11]:
set.seed(202)

x <- rnorm(50)
y <- x + rnorm(50, sd = 0.5)

# 相関係数と検定
cor.test(x, y)


	Pearson's product-moment correlation

data:  x and y
t = 14.856, df = 48, p-value < 2.2e-16
alternative hypothesis: true correlation is not equal to 0
95 percent confidence interval:
 0.8397840 0.9459944
sample estimates:
      cor 
0.9062857 


## 7. 回帰分析（lm）

ここでは単回帰分析の例を示します。

In [12]:
# 単回帰分析（y を x から説明）
fit_lm <- lm(y ~ x)
summary(fit_lm)


Call:
lm(formula = y ~ x)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.18281 -0.25941  0.01854  0.28746  1.09043 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  0.01062    0.06889   0.154    0.878    
x            0.98323    0.06619  14.856   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4812 on 48 degrees of freedom
Multiple R-squared:  0.8214,	Adjusted R-squared:  0.8176 
F-statistic: 220.7 on 1 and 48 DF,  p-value: < 2.2e-16


### 7-1. ロジスティック回帰（glm, family = binomial）

説明変数から 0/1 の目的変数を予測する場合に使います。

In [13]:
set.seed(303)

x2 <- rnorm(100)
p  <- 1 / (1 + exp(-x2))  # ロジスティック関数
y2 <- rbinom(100, size = 1, prob = p)

fit_logit <- glm(y2 ~ x2, family = binomial)
summary(fit_logit)


Call:
glm(formula = y2 ~ x2, family = binomial)

Deviance Residuals: 
    Min       1Q   Median       3Q      Max  
-1.8828  -1.0174   0.5467   0.9307   1.7259  

Coefficients:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)  0.08472    0.21949   0.386  0.69950    
x2           0.96886    0.25865   3.746  0.00018 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 138.27  on 99  degrees of freedom
Residual deviance: 120.53  on 98  degrees of freedom
AIC: 124.53

Number of Fisher Scoring iterations: 3


## 8. 考察（演習）

ここまでで学んだ統計テスト（t検定、χ²検定、分散分析、相関・回帰など）を用いて、
自分でデータを生成したり、与えられたデータを分析してみましょう。

例：

- グループ数や平均値・標準偏差を変えて再実行する
- サンプルサイズを変えて、p値がどのように変化するかを観察する
- ロジスティック回帰の説明変数を 2 つ以上に増やしてみる

分析の結果について、簡単なコメントを書いておくと復習に役立ちます。